# `financialtools.processor` — User Guide

> **Note:** As of the L1 refactor, implementation lives in two focused modules:
> - `financialtools/downloader.py` — `Downloader` class
> - `financialtools/evaluator.py` — `FundamentalMetricsEvaluator` + module-level helpers
>
> `financialtools/processor.py` is now a re-export shim — all imports below continue to work unchanged.

Core data-acquisition and metric-evaluation module: fetches raw financials from Yahoo Finance,
reshapes them into long-format DataFrames, computes 24 scored metrics plus 14 unscored
extended metrics, scores the 24 on a 1–5 scale, and detects red flags.

```
financialtools/processor.py   (re-export shim → downloader.py + evaluator.py)
│
├── RateLimiter(per_minute, per_hour, per_day)
│   └── acquire()               block until a call slot is available
│
├── Downloader(ticker)
│   ├── from_ticker(ticker)     classmethod — fetch + reshape; raises DownloadError on failure
│   ├── get_merged_data()       long-format DataFrame (balance sheet + income + cashflow)
│   ├── get_info_data()         DataFrame with marketCap, forwardPE, etc.
│   ├── combine_merged_data()   classmethod — concat across multiple Downloader instances
│   ├── combine_info_data()     classmethod — concat info DataFrames
│   └── stream_download()       classmethod — yield Downloaders one by one, save to Parquet
│
├── FundamentalMetricsEvaluator(data, weights)   ← preferred name
│   ├── evaluate()              full pipeline → dict with 6 keys
│   ├── compute_metrics()       compute 24 SCORED_METRICS columns
│   ├── compute_extended_metrics()  14 unscored metrics (efficiency, growth, red-flag ratios)
│   ├── compute_valuation_metrics()  P/E, P/B, P/FCF, EarningsYield, FCFYield
│   ├── compute_scores()        score each metric 1–5
│   ├── raw_red_flags()         cash-flow quality flags
│   ├── _score_metric()         internal — score a long-format metrics DataFrame
│   └── _metrics_red_flags()    internal — threshold-based flag detection
│
└── Module-level helpers
    ├── SCORED_METRICS          list[str] — 24 metric column names
    ├── _EMPTY_RESULT_KEYS      tuple[str] — canonical evaluate() output keys (6)
    └── _empty_result()         factory → {k: pd.DataFrame() for k in _EMPTY_RESULT_KEYS}
        (also exported as empty_evaluate_result() from financialtools public API)
```

> **Deprecated:** `FundamentalTraderAssistant` is a backward-compat alias for
> `FundamentalMetricsEvaluator`. It emits `DeprecationWarning` on construction.
> Use `FundamentalMetricsEvaluator` in new code.

| Symbol | Needs yfinance | Needs LLM | Best for |
|---|---|---|---|
| `RateLimiter` | no | no | throttling any external API call loop |
| `Downloader.from_ticker` | **yes** | no | fetching one ticker's financials |
| `Downloader.get_merged_data` | no | no | accessing already-fetched data |
| `FundamentalMetricsEvaluator` | no | no | scoring metrics from any merged DataFrame |
| `empty_evaluate_result()` | no | no | safe fallback / test scaffolding |

**Prerequisites:**
- Sections 1–3 and 7–9 run with **no API key** and **no internet connection**.
- Sections 4–6 require `yfinance` and an active internet connection.

## Sections
1. [Setup](#1-setup)
2. [Module-level constants](#2-module-level-constants)
3. [RateLimiter](#3-ratelimiter)
4. [FundamentalMetricsEvaluator — construction](#4-fundamentalmetricsevaluator--construction)
5. [FundamentalMetricsEvaluator — evaluate()](#5-fundamentalmetricsevaluator--evaluate)
6. [Downloader (Live)](#6-downloader-live)
7. [Error handling](#7-error-handling)
8. [Common failure modes](#8-common-failure-modes)
9. [End-to-end pattern](#9-end-to-end-pattern)

## 1 â€” Setup

In [ ]:
import logging
import numpy as np
import pandas as pd

from financialtools.processor import (
    RateLimiter,
    Downloader,
    FundamentalMetricsEvaluator,
    SCORED_METRICS,
    _EMPTY_RESULT_KEYS,
)
from financialtools.exceptions import DownloadError, EvaluationError

# empty_evaluate_result is the public alias for _empty_result()
from financialtools import empty_evaluate_result

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    force=True,
)
print("Imports OK")

## 2 — Module-level constants

`SCORED_METRICS` is a reference list of the **24** metric column names produced by
`compute_metrics()` (original 11 + 13 extended scored metrics). It is kept for
documentation and test-scaffolding purposes.
`evaluate()` and `compute_scores()` **derive `value_vars` dynamically** from
`compute_metrics()` output columns (all columns that are not `ticker`, `time`, or `sector`),
so adding a new metric to `compute_metrics()` automatically includes it in scoring without
editing `SCORED_METRICS`.

`_EMPTY_RESULT_KEYS` is the tuple of **6** keys that every `evaluate()` return value must
contain. `empty_evaluate_result()` (public) / `_empty_result()` (internal) is the factory
that creates a fresh dict of empty DataFrames with exactly those keys.

> Always call `empty_evaluate_result()` — never copy `_EMPTY_RESULT` directly — because a
> shallow copy would share DataFrame objects across calls.

In [ ]:
print(f"SCORED_METRICS ({len(SCORED_METRICS)} items):")
for m in SCORED_METRICS:
    print(f"  {m}")

print(f"\n_EMPTY_RESULT_KEYS : {_EMPTY_RESULT_KEYS}")

er = empty_evaluate_result()
print(f"\nempty_evaluate_result() keys  : {list(er.keys())}")
print(f"All values empty              : {all(v.empty for v in er.values())}")
print(f"Distinct objects              : {len({id(v) for v in er.values()})} (should equal {len(er)})")

## 3 â€” RateLimiter

`RateLimiter` is a thread-safe sliding-window rate limiter. It tracks call timestamps and
sleeps the calling thread until all three windows (minute, hour, day) have a free slot.

### Constructor

```python
RateLimiter(per_minute=60, per_hour=360, per_day=8000)
```

| Parameter | Default | Meaning |
|---|---|---|
| `per_minute` | 60 | max calls within any 60-second window |
| `per_hour` | 360 | max calls within any 3600-second window |
| `per_day` | 8000 | max calls within any 86400-second window |

### acquire()

Blocks until all three windows have a free slot, then records the timestamp. The internal
`calls` list is pruned to the last 24 hours on every `acquire()` call to bound memory usage.

### Thread safety

All state mutations (reading + writing `self.calls`) are protected by `threading.Lock`.
It is safe to share one `RateLimiter` instance across threads â€” for example across a
`ThreadPoolExecutor` that downloads multiple tickers in parallel.

### Typical usage

Pass a `RateLimiter` instance to `Downloader.stream_download()` to pace bulk downloads.

In [3]:
import time
import threading

# --- Basic construction ---
limiter = RateLimiter(per_minute=10, per_hour=100, per_day=1000)
print(f"per_minute : {limiter.per_minute}")
print(f"per_hour   : {limiter.per_hour}")
print(f"per_day    : {limiter.per_day}")
print(f"calls so far: {len(limiter.calls)}")

# --- acquire() under the limit â€” should return immediately ---
for i in range(3):
    limiter.acquire()
    print(f"  call {i+1}: acquired at {time.strftime('%H:%M:%S')}")

print(f"calls recorded: {len(limiter.calls)}")

per_minute : 10
per_hour   : 100
per_day    : 1000
calls so far: 0
  call 1: acquired at 16:56:05
  call 2: acquired at 16:56:05
  call 3: acquired at 16:56:05
calls recorded: 3


In [4]:
# --- Thread-safety demo: 5 threads sharing one RateLimiter with per_minute=5 ---
# Each thread calls acquire() once. With per_minute=5, the first 5 calls go through
# immediately; a 6th would block. Here we fire exactly 5 to show safe concurrent use.

results = []
lock = threading.Lock()

def worker(tid):
    limiter_shared.acquire()
    with lock:
        results.append(f"thread-{tid} acquired at {time.strftime('%H:%M:%S')}")

limiter_shared = RateLimiter(per_minute=5, per_hour=100, per_day=1000)
threads = [threading.Thread(target=worker, args=(i,)) for i in range(5)]
for t in threads:
    t.start()
for t in threads:
    t.join()

for r in sorted(results):
    print(r)
print(f"Total calls recorded: {len(limiter_shared.calls)}")

thread-0 acquired at 16:56:09
thread-1 acquired at 16:56:09
thread-2 acquired at 16:56:09
thread-3 acquired at 16:56:09
thread-4 acquired at 16:56:09
Total calls recorded: 5


## 4 — FundamentalMetricsEvaluator — construction

`FundamentalMetricsEvaluator` validates its inputs eagerly in `__init__` and raises
`EvaluationError` for any of four guard conditions:

| Guard | Raises when |
|---|---|
| Missing or empty data | `data` is empty or has no `'ticker'` column |
| Empty ticker | `data['ticker'].dropna().unique()` is empty |
| Multiple tickers | `data` contains more than one distinct non-null ticker value |
| Empty sector in weights | `weights['sector'].dropna().unique()` is empty |

All three attributes start as empty DataFrames:

```python
self.metrics       = pd.DataFrame()
self.eval_metrics  = pd.DataFrame()
self.scores        = pd.DataFrame()
```

They are populated lazily by `compute_metrics()`, `compute_valuation_metrics()`, and
`evaluate()`.

### Two ways to construct

**Pass a weights DataFrame** (full control):
```python
from financialtools.analysis import build_weights
weights = build_weights("technology")
fme = FundamentalMetricsEvaluator(data=merged, weights=weights)
```

**Pass a sector name** (preferred shorthand via `FundamentalEvaluator` wrapper):
```python
from financialtools.wrappers import FundamentalEvaluator
evaluator = FundamentalEvaluator(df=merged, sector="technology")
result = evaluator.evaluate_single("AAPL")
```

### Live data via Downloader

The cells below download real AAPL financials using `Downloader.from_ticker()`.
`get_merged_data()` returns balance-sheet + income-statement + cash-flow joined on
`(ticker, time)`, **and automatically broadcasts market-data columns** (`marketcap`,
`currentprice`, `sharesoutstanding`) from `_info` across all time periods.
No manual `get_info_data()` merge is required before constructing `FundamentalMetricsEvaluator`.
Sector weights come from `config.sec_sector_metric_weights`.

> **Requires internet.** `from_ticker()` raises `DownloadError` on failure — always
> catch it or let it propagate. Do not check `merged.empty` as the only failure signal.

### Weight dict key convention

Two weight dicts are available in `config.py`:

| Dict | Key format | Example | Use when |
|---|---|---|---|
| `sec_sector_metric_weights` | yfinance `sectorKey` (lowercase) | `"technology"`, `"financial-services"` | Building from a `Downloader` result (**canonical**) |
| `sector_metric_weights` | Display name (title case) | `"Technology Services"` | **Legacy** — prefer `sec_sector_metric_weights` |

Use `list_sectors()` to enumerate all valid `sec_sector_metric_weights` keys:
```python
from financialtools import list_sectors
print(list_sectors())
```

In [ ]:
# Download AAPL financials — requires yfinance and an active internet connection.
# from_ticker() raises DownloadError on failure — catch it if you need a soft-failure path.
import re

TICKER = "AAPL"

try:
    d = Downloader.from_ticker(TICKER)
except DownloadError as e:
    raise RuntimeError(f"Download failed for {TICKER}: {e}") from e

merged = d.get_merged_data()

# Derive sector from yfinance info — single source of truth (sec_sector_metric_weights convention).
SECTOR = d.get_info_data()["sector"].str.lower().to_string(index=False)
SECTOR = re.sub(" ", "-", SECTOR)

print(f"merged shape : {merged.shape}")   # (time_periods, financial_columns)
print(f"sector       : {SECTOR}")

if merged.empty:
    raise RuntimeError(f"get_merged_data() returned empty for {TICKER} — check logs/error.log")

merged[["ticker", "time", "total_revenue", "gross_profit", "free_cash_flow"]].head()

In [6]:
# get_merged_data() automatically merges marketcap, currentprice, and sharesoutstanding
# from _info across all time periods â€” no manual enrichment needed.
print(f"data shape : {merged.shape}")

market_cols = [c for c in Downloader._MARKET_COLS if c in merged.columns]
missing     = [c for c in Downloader._MARKET_COLS if c not in merged.columns]
if missing:
    print(f"Warning: market columns not found in merged data: {missing}")

print(f"Market columns present: {market_cols}")
merged[["ticker", "time"] + market_cols].head()

data shape : (5, 169)
Market columns present: ['marketcap', 'currentprice', 'sharesoutstanding']


,ticker,time,marketcap,currentprice,sharesoutstanding
0,AAPL,2021-09-30,3848063942656,261.82,14681140000
1,AAPL,2022-09-30,3848063942656,261.82,14681140000
2,AAPL,2023-09-30,3848063942656,261.82,14681140000
3,AAPL,2024-09-30,3848063942656,261.82,14681140000
4,AAPL,2025-09-30,3848063942656,261.82,14681140000


In [ ]:
from financialtools.config import sec_sector_metric_weights

# Build the weights DataFrame from the canonical sector config — single source of truth.
# AAPL is in "technology" per sec_sector_metric_weights (yfinance sectorKey convention).
sector_weights_dict = sec_sector_metric_weights[SECTOR]
weights = pd.DataFrame({
    "sector" : SECTOR,
    "metrics": list(sector_weights_dict.keys()),
    "weights": list(sector_weights_dict.values()),
})

print(f"weights shape : {weights.shape}")

# Construct the evaluator — merged already includes marketcap, currentprice,
# sharesoutstanding from get_merged_data(); no separate info merge required.
fme = FundamentalMetricsEvaluator(data=merged, weights=weights)
print(f"\nfme.ticker  : {fme.ticker}")
print(f"fme.sector  : {fme.sector}")
print(f"fme.metrics (before evaluate) empty : {fme.metrics.empty}")
print(f"fme.scores  (before evaluate) empty : {fme.scores.empty}")

## 5 — FundamentalMetricsEvaluator — evaluate()

`evaluate()` runs the full scoring pipeline and returns a dict with **six** keys:

| Key | Type | Content |
|---|---|---|
| `metrics` | `pd.DataFrame` | wide — one row per `(ticker, time)`, 24 scored metric columns |
| `eval_metrics` | `pd.DataFrame` | wide — valuation metrics: P/E, P/B, P/FCF, EarningsYield, FCFYield |
| `composite_scores` | `pd.DataFrame` | one row per `(sector, ticker, time)` with `composite_score` |
| `raw_red_flags` | `pd.DataFrame` | cash-flow quality flags (negative FCF/OCF, EBITDA >> OCF) |
| `red_flags` | `pd.DataFrame` | threshold-based flags (negative margins, high D/E, etc.) |
| `extended_metrics` | `pd.DataFrame` | 14 unscored columns: efficiency chain, growth rates, red-flag ratios |

### Failure behaviour: raises EvaluationError

`evaluate()` raises `EvaluationError` on any internal failure. Callers that need a
soft-failure path should catch it and call `empty_evaluate_result()` themselves:

```python
from financialtools.exceptions import EvaluationError
from financialtools import empty_evaluate_result

try:
    result = fme.evaluate()
except EvaluationError as e:
    logger.error("evaluate failed: %s", e)
    result = empty_evaluate_result()
```

### Composite score formula

```
composite_score = sum(score_i * weight_i) / sum(weight_i)
```

where `score_i` is the 1–5 score for metric `i` (from `_score_metric()`) and `weight_i`
comes from the `weights` DataFrame passed to the constructor.

### Scored vs unscored split

**24 scored metrics** feed the composite score. **14 unscored extended metrics**
(returned under `extended_metrics`) do not — they are time-differential (`pct_change`) or
derived chains (e.g. CCC) that lack universal thresholds. Four scored metrics use
**inverse scoring** (lower value → higher score): `DebtToEquity`, `DebtRatio`,
`NetDebtToEBITDA`, `CapexRatio`.

In [ ]:
result = fme.evaluate()

print("Result keys:", list(result.keys()))
print(f"All keys present: {set(result.keys()) == set(_EMPTY_RESULT_KEYS)}")

print(f"\n--- metrics ({result['metrics'].shape}) ---")
print(result["metrics"][["ticker", "time", "GrossMargin", "ROE", "DebtToEquity"]].to_string(index=False))

print(f"\n--- eval_metrics ({result['eval_metrics'].shape}) ---")
print(result["eval_metrics"][["ticker", "time", "P/E", "P/B", "FCFYield"]].to_string(index=False))

print(f"\n--- composite_scores ({result['composite_scores'].shape}) ---")
print(result["composite_scores"].to_string(index=False))

print(f"\n--- extended_metrics ({result['extended_metrics'].shape}) ---")
ext = result["extended_metrics"]
print(ext[["ticker", "time", "DSO", "DIO", "DPO", "CCC", "RevenueGrowth"]].to_string(index=False))

In [ ]:
# Inspect red flags from real AAPL data.
raw_rf = result["raw_red_flags"]
rf     = result["red_flags"]

print(f"raw_red_flags rows : {len(raw_rf)}")
if not raw_rf.empty:
    print(raw_rf.to_string(index=False))
else:
    print("  (none — all cash flows positive)")

print(f"\nred_flags rows     : {len(rf)}")
if not rf.empty:
    print(rf.to_string(index=False))
else:
    print("  (none — all metrics within thresholds)")

# Force a red-flag scenario by injecting bad values into the two earliest rows.
data_bad = merged.copy().sort_values("time").reset_index(drop=True)
data_bad.loc[0, "free_cash_flow"] = -500_000_000
data_bad.loc[1, "net_income_common_stockholders"] = -200_000_000

fme_bad    = FundamentalMetricsEvaluator(data=data_bad, weights=weights)
result_bad = fme_bad.evaluate()

print("\n--- raw_red_flags with injected negative FCF ---")
print(result_bad["raw_red_flags"].to_string(index=False))

print("\n--- red_flags with injected negative net margin ---")
print(result_bad["red_flags"].to_string(index=False))

## 6 — Downloader (Live)

> **Requires internet.**  All cells in this section call `yfinance`.

### Downloading multiple tickers

`combine_merged_data()` and `combine_info_data()` are classmethods that concatenate
results across multiple `Downloader` instances into a single DataFrame.

```python
tickers = ["AAPL", "MSFT", "GOOGL"]
downloaders = [Downloader.from_ticker(t) for t in tickers]
merged_all = Downloader.combine_merged_data(downloaders)
info_all   = Downloader.combine_info_data(downloaders)
```

Failed tickers return an empty `Downloader`; `combine_merged_data()` silently skips
empty instances — the combined result contains only successful tickers.

### Streaming with rate-limiting

`stream_download()` is a classmethod generator that yields one `Downloader` per ticker
and optionally saves each to Parquet.  Pass a `RateLimiter` to pace bulk downloads.

```python
limiter = RateLimiter(per_minute=10, per_hour=100, per_day=1000)
for dl in Downloader.stream_download(tickers, rate_limiter=limiter, save_parquet=False):
    if not dl.get_merged_data().empty:
        print(f"  {dl.ticker}: {dl.get_merged_data().shape}")
```

> `save_parquet=True` writes `<ticker>.parquet` to the current working directory.

In [10]:
# Download two tickers and combine — requires internet.
TICKERS = ["AAPL", "MSFT"]

downloaders  = [Downloader.from_ticker(t) for t in TICKERS]
merged_multi = Downloader.combine_merged_data(downloaders)
info_multi   = Downloader.combine_info_data(downloaders)

print(f"combined merged shape : {merged_multi.shape}")
print(f"combined info   shape : {info_multi.shape}")
print(f"tickers in merged     : {sorted(merged_multi['ticker'].unique())}")

merged_multi[["ticker", "time", "total_revenue", "free_cash_flow"]].dropna().tail(6)

combined merged shape : (10, 202)
combined info   shape : (2, 180)
tickers in merged     : ['AAPL', 'MSFT']


,ticker,time,total_revenue,free_cash_flow
3,AAPL,2024-09-30,3.910350e+11,1.088070e+11
4,AAPL,2025-09-30,4.161610e+11,9.876700e+10
6,MSFT,2022-06-30,1.982700e+11,6.514900e+10
7,MSFT,2023-06-30,2.119150e+11,5.947500e+10
8,MSFT,2024-06-30,2.451220e+11,7.407100e+10
9,MSFT,2025-06-30,2.817240e+11,7.161100e+10


In [ ]:
# stream_download() — iterate one ticker at a time with rate limiting.
# Signature: stream_download(tickers, limiter, out_dir="financial_data")
# out_dir is relative to the caller's cwd — pass an absolute path from notebooks.
import tempfile, os

limiter_bulk = RateLimiter(per_minute=10, per_hour=100, per_day=1000)

with tempfile.TemporaryDirectory() as tmp:
    for dl in Downloader.stream_download(TICKERS, limiter_bulk, out_dir=tmp):
        m = dl.get_merged_data()
        status = f"{m.shape[0]} rows" if not m.empty else "EMPTY (check logs/error.log)"
        print(f"  {dl.ticker}: {status}")

## 7 — Error handling

> **No internet required.**

### `EvaluationError` — raised by `__init__`

`FundamentalMetricsEvaluator.__init__` is the only place `EvaluationError` is raised.
It validates two things:

1. `data` must be non-empty and contain a `'ticker'` column with exactly one non-null value.
2. `weights` must contain exactly one non-null sector value.

Any internal failure inside `evaluate()` raises `EvaluationError` — it propagates to the caller.
Catch it explicitly and call `empty_evaluate_result()` to obtain the safe-fallback shape.

### `empty_evaluate_result()` as the safe-fallback shape

```python
from financialtools.exceptions import EvaluationError
from financialtools import empty_evaluate_result

try:
    result = fme.evaluate()
except EvaluationError as e:
    logger.warning("evaluate failed: %s", e)
    result = empty_evaluate_result()

if result["composite_scores"].empty:
    # evaluation produced no scores — check logs/error.log
    ...
```

### `DownloadError`

`DownloadError` is raised by `Downloader.from_ticker()` when yfinance fails.
Always catch it at the call site if you need a soft-failure path:

```python
from financialtools.exceptions import DownloadError

try:
    d = Downloader.from_ticker(ticker)
except DownloadError as e:
    logger.warning("Download failed: %s", e)
    continue   # or return empty_evaluate_result()
```

`DownloaderWrapper.download_data()` also raises `DownloadError` when all tickers fail.

In [ ]:
from financialtools.exceptions import EvaluationError
from financialtools.config import sec_sector_metric_weights

_weights_test = pd.DataFrame({
    "sector" : "technology",
    "metrics": list(sec_sector_metric_weights["technology"].keys()),
    "weights": list(sec_sector_metric_weights["technology"].values()),
})

# ── Guard 1: empty data raises EvaluationError ───────────────────────────────
try:
    FundamentalMetricsEvaluator(data=pd.DataFrame(), weights=_weights_test)
    print("ERROR: should have raised")
except EvaluationError as e:
    print(f"EvaluationError (empty data): {e}")

# ── Guard 2: data missing 'ticker' column raises EvaluationError ─────────────
try:
    FundamentalMetricsEvaluator(data=pd.DataFrame({"time": ["2023"]}), weights=_weights_test)
    print("ERROR: should have raised")
except EvaluationError as e:
    print(f"EvaluationError (no ticker col): {e}")

# ── Guard 3: multiple tickers raises EvaluationError ─────────────────────────
_multi = pd.DataFrame({"ticker": ["AAPL", "MSFT"], "time": ["2023", "2023"]})
try:
    FundamentalMetricsEvaluator(data=_multi, weights=_weights_test)
    print("ERROR: should have raised")
except EvaluationError as e:
    print(f"EvaluationError (multi-ticker): {e}")

# ── Guard 4: evaluate() never raises — returns empty_evaluate_result() on failure
_single = pd.DataFrame({"ticker": ["AAPL"], "time": ["2023"]})   # missing all metric cols
fme_minimal = FundamentalMetricsEvaluator(data=_single, weights=_weights_test)
result_minimal = fme_minimal.evaluate()

print(f"\nevaluate() on minimal data returned {len(result_minimal)} keys")
print(f"composite_scores empty: {result_minimal['composite_scores'].empty}  <- expected True")
print(f"All values empty      : {all(v.empty for v in result_minimal.values())}")

In [ ]:
from typing import Optional

def safe_composite_score(ticker: str, fme_inst: FundamentalMetricsEvaluator) -> Optional[float]:
    """Return the latest composite score, or None if evaluation failed."""
    result = fme_inst.evaluate()
    cs = result["composite_scores"]
    if cs.empty:
        print(f"[{ticker}] evaluation produced no scores — check logs/error.log")
        return None
    return float(cs.sort_values("time")["composite_score"].iloc[-1])

# Demonstrate with valid fme (if section 4 was run):
try:
    score = safe_composite_score(fme.ticker, fme)
    if score is not None:
        print(f"Latest composite score for {fme.ticker}: {score:.4f}")
except NameError:
    print("Run section 4 first to define 'fme'")

## 8 — Common failure modes

> **No internet required.**  Reference table — no code to run.

| Symptom | Likely cause | Where to look / fix |
|---|---|---|
| `EvaluationError` at construction | `data` is empty, missing `'ticker'` column, or has >1 unique tickers; or `weights` has 0 or >1 unique sectors | Confirm `data["ticker"].nunique() == 1` and `'ticker' in data.columns` before constructing |
| `composite_scores` is empty | Internal failure in `evaluate()` | `logs/error.log` — search for `evaluate() failed` |
| All extended-metric columns NaN | Optional source column absent (e.g. `inventory`, `invested_capital`, `ebit`) | Logged as WARNING; inspect `merged.columns` |
| Growth rates in wrong order | `time` values not sortable as strings | `compute_extended_metrics()` sorts by `time`; verify `time` column format |
| `KeyError` on sector | Wrong weight dict for the key format | Use `sec_sector_metric_weights` for yfinance `sectorKey`; `sector_metric_weights` is legacy |
| `marketcap` / `currentprice` all NaN | `_info` unavailable for ticker | Logged as WARNING; valuation metrics will be NaN |
| `extended_metrics` key missing from result | `_EMPTY_RESULT_KEYS` not updated | Add the key to `_EMPTY_RESULT_KEYS` in `evaluator.py` |
| `SectorNotFoundError` in `chains.py` | Ticker absent from `sector_ticker.txt` | Add ticker to mapping file; regenerate benchmark files |
| `DownloadError` from `from_ticker()` | yfinance network failure or bad ticker | Catch `DownloadError` and log; `logs/error.log` has per-ticker details |
| `DownloadError` from `download_data()` | All tickers in the batch failed | Check `logs/error.log`; previously returned `None`, now raises |
| Logs written to wrong directory | `wrappers.py` copied out of package | Always import from the installed package; `_LOGS_DIR` is `__file__`-relative |
| LLM returns malformed JSON | Model deviation from schema | `invoke_chain()` retries once with a fix prompt; check `logs/error.log` |

## 9 — End-to-end pattern

> **Requires internet** for the download step.  Evaluation and inspection are offline.

Full pipeline from ticker symbol to scored results:

```
Ticker
  → Downloader.from_ticker()
  → get_merged_data()          # long-format; market cols auto-broadcast
  → FundamentalMetricsEvaluator(data, weights)
  → evaluate()                 # 6-key dict
      metrics, eval_metrics, composite_scores,
      raw_red_flags, red_flags, extended_metrics
```

The cell below runs the complete pipeline for a single ticker using only the
`processor` module — no `wrappers.py` dependency.

In [ ]:
# ── End-to-end: single ticker, no wrappers dependency ────────────────────
from financialtools.config import sec_sector_metric_weights
from financialtools.exceptions import DownloadError

TICKER_E2E = "AAPL"
SECTOR_E2E = "technology"

# 1 — Download
try:
    dl_e2e = Downloader.from_ticker(TICKER_E2E)
except DownloadError as e:
    raise RuntimeError(f"Download failed for {TICKER_E2E}: {e}") from e

merged_e2e = dl_e2e.get_merged_data()
if merged_e2e.empty:
    raise RuntimeError(f"get_merged_data() returned empty for {TICKER_E2E} — check logs/error.log")
print(f"[1] merged shape : {merged_e2e.shape}")

# 2 — Build weights
sector_dict = sec_sector_metric_weights[SECTOR_E2E]
weights_e2e = pd.DataFrame({
    "sector" : SECTOR_E2E,
    "metrics": list(sector_dict.keys()),
    "weights": list(sector_dict.values()),
})
print(f"[2] weights shape: {weights_e2e.shape}")

# 3 — Evaluate
fme_e2e    = FundamentalMetricsEvaluator(data=merged_e2e, weights=weights_e2e)
result_e2e = fme_e2e.evaluate()

print(f"[3] result keys  : {list(result_e2e.keys())}")
print(f"    metrics shape: {result_e2e['metrics'].shape}")
print(f"    ext metrics  : {result_e2e['extended_metrics'].shape}")

# 4 — Composite score
cs = result_e2e["composite_scores"]
if not cs.empty:
    latest = cs.sort_values("time").iloc[-1]
    print(f"\n[4] Latest composite score ({latest['time']}) : {latest['composite_score']:.4f}")

# 5 — Revenue growth
ext = result_e2e["extended_metrics"].sort_values("time")
if not ext.empty:
    print("\n[5] Revenue growth by year:")
    print(ext[["time", "RevenueGrowth"]].dropna().to_string(index=False))